In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras import datasets, layers, models

from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

import os
import logging
import warnings

#logging.getLogger("tensorflow").setLevel(logging.ERROR)
#os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
warnings.filterwarnings('ignore')

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar100.load_data()

x_train = x_train.astype("float32") / 255.0
x_test  = x_test.astype("float32")  / 255.0


169001437/169001437 ━━━━━━━━━━━━━━━━━━━━ 13s 0us/step


# ResNet Style

In [ ]:
from tensorflow.keras import Model, Input
from tensorflow.keras.layers import (Conv2D, BatchNormalization, Activation, Add,
                                     GlobalAveragePooling2D, Dense, Dropout,
                                     GlobalMaxPooling2D, Reshape, Multiply)

def create_improved_model():
    from tensorflow.keras import layers, Model, Input

    def residual_block(x, filters, stride=1):
        skip = x
        # Pre-activation
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
        x = layers.Conv2D(filters, 3, strides=stride, padding="same", use_bias=False)(x)

        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
        x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)

        if stride != 1 or skip.shape[-1] != filters:
            skip = layers.Conv2D(filters, 1, strides=stride, padding="same", use_bias=False)(skip)

        return layers.Add()([x, skip])

    inputs = Input((32, 32, 3))
    x = layers.Conv2D(64, 3, padding="same", use_bias=False)(inputs)

    # Stacks residuais
    for filters, reps in [(64, 2), (128, 2), (256, 2)]:
        for i in range(reps):
            stride = 2 if i == 0 and filters != 64 else 1
            x = residual_block(x, filters, stride)

    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(100, activation="softmax")(x)

    return Model(inputs, outputs)

In [ ]:
# -------------------------------------------------------------------
# 2) Helpers
# -------------------------------------------------------------------
def get_callbacks():
        return [
            EarlyStopping(monitor=MONITOR_MET, patience=PATIENCE_ES, verbose=0, restore_best_weights=True),
            ModelCheckpoint(MODEL_PATH, monitor=MONITOR_MET, save_best_only=True, verbose=0),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7
            )
        ]
def plot_history(history):
    plt.figure(figsize=(12,4))

    # Loss
    plt.subplot(1,2,1)
    plt.plot(history.history["loss"], label="train")
    plt.plot(history.history[MONITOR_MET], label="val")
    plt.title("Loss")
    plt.legend()

    # Accuracy
    plt.subplot(1,2,2)
    plt.plot(history.history["accuracy"], label="train")
    plt.plot(history.history["val_accuracy"], label="val")
    plt.title("Accuracy")
    plt.legend()

    plt.show()

def evaluate_model(model, x_test, y_test):
    loss, acc = model.evaluate(x_test, y_test, verbose=0)
    print(f"Acurácia final no test set: {acc:.4f}")
    return loss, acc

def export_tflite(model, filename="cifar100_simple.tflite"):
    print(f"Convertendo modelo para TFLite em `{filename}`...")
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    tflite_model = converter.convert()
    with open(filename, "wb") as f:
        f.write(tflite_model)
    print("✅ Export concluído.")
    #return tflite_model

In [ ]:
# -------------------------------------------------------------------
# 1) Configurações gerais (hiperparâmetros, caminhos, etc)
# -------------------------------------------------------------------

EPOCHS       = 50
BATCH_SIZE   = 128
VALID_SPLIT  = 0.2
MODEL_PATH   = "best_cifar100_model.h5"
TFLITE_PATH  = "cifar100_simple.tflite"
PATIENCE_ES  = 10
MONITOR_MET  = "val_loss"

In [ ]:
# -------------------------------------------------------------------
# 3) O seu main, orquestrando tudo
# -------------------------------------------------------------------
# 3.1 Cria, compila e mostra sumário
print("🔨 Criando e compilando modelo para CIFAR-100")
model = create_improved_model()

datagen = tf.keras.preprocessing.image.ImageDataGenerator(
rotation_range=15,
width_shift_range=0.1,
height_shift_range=0.1,
horizontal_flip=True,
zoom_range=0.1
)
datagen.fit(x_train)


model.compile(
optimizer=tf.keras.optimizers.AdamW(learning_rate=0.001, weight_decay=0.0001),
loss="sparse_categorical_crossentropy",
metrics=["accuracy"]
)
model.summary()

# 3.2 Treina com callbacks
print(f"\n🚂 Iniciando treino por {EPOCHS} epochs (batch size={BATCH_SIZE})")
history = model.fit(
datagen.flow(x_train, y_train, batch_size=BATCH_SIZE),
steps_per_epoch=len(x_train) // BATCH_SIZE,
epochs=EPOCHS,
validation_data=(x_test, y_test),  # ou use validation_split se preferir
callbacks=get_callbacks(),
verbose=1
)

# 3.3 Exibe gráficos de evolução
print("\n📊 Plotando curva de treino")
plot_history(history)

# 3.4 Avaliação final
print("\n🔍 Avaliando no conjunto de teste")
evaluate_model(model, x_test, y_test)

# 3.5 Exporta para TFLite
export_tflite(model, TFLITE_PATH)

In [ ]:
model.summary()
plot_history(history)
evaluate_model(model, x_test, y_test)

# quantizar

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os


QUANTIZED_MODEL_FILENAME = "/content/cifar100_simple_int8.tflite"

def convert_to_tflite_quantized(keras_model, representative_data, filename):
    print("Converting to TFLite with full integer quantization...")

    def representative_dataset_gen():
        for i in range(len(representative_data)):
            yield [representative_data[i:i+1]]

    converter = tf.lite.TFLiteConverter.from_keras_model(keras_model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = representative_dataset_gen
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8

    tflite_quant_model = converter.convert()

    with open(filename, "wb") as f:
        f.write(tflite_quant_model)

    size_kb = os.path.getsize(filename) / 1024
    print(f"Quantized model saved to {filename} ({size_kb:.1f} KB)")
    return tflite_quant_model

def run_quantized_inference(tflite_model_content, float_image_sample):
    interpreter = tf.lite.Interpreter(model_content=tflite_model_content)
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    scale, zero_point = input_details["quantization"]
    quantized_input = (float_image_sample / scale + zero_point).astype(input_details["dtype"])

    interpreter.set_tensor(input_details["index"], quantized_input)
    interpreter.invoke()

    quantized_output = interpreter.get_tensor(output_details["index"])
    out_scale, out_zero_point = output_details["quantization"]
    float_output = (quantized_output.astype(np.float32) - out_zero_point) * out_scale

    predicted_label = np.argmax(float_output)
    return predicted_label

def plot_prediction(image, ):
    plt.figure(figsize=(4, 4))
    plt.imshow(image)
    plt.axis('off')
    plt.show()

def main():
    # This script assumes a trained Keras model file 'best_cifar10_model.keras'
    # and the CIFAR-10 dataset are available.
    try:
        model = tf.keras.models.load_model('/content/best_cifar100_model.h5')
    except (IOError, ImportError) as e:
        print(f"Error: Could not load 'best_cifar10_model.keras'. {e}")
        print("Please ensure the model has been trained and saved first.")
        return


    representative_data = x_train[:200]
    tflite_model_content = convert_to_tflite_quantized(
        keras_model=model,
        representative_data=representative_data,
        filename=QUANTIZED_MODEL_FILENAME
    )

    print("\nVerifying inference with the INT8 model...")





    test_index = 15


    image_sample = x_test[test_index: test_index + 1]
    true_label = int(y_test[test_index][0])


    predicted_label = run_quantized_inference(tflite_model_content, image_sample)


    print(f"True Class:     {true_label} ")
    print(f"Predicted Class:  {predicted_label} ")

    plot_prediction(image_sample[0])

if __name__ == "__main__":
    main()

In [ ]:
x_test

array([[[[0.78039217, 0.84313726, 0.9764706 ],
         [0.76862746, 0.827451  , 0.95686275],
         [0.7647059 , 0.8235294 , 0.9529412 ],
         ...,
         [0.84705883, 0.90588236, 0.98039216],
         [0.8509804 , 0.90588236, 0.98039216],
         [0.8784314 , 0.91764706, 0.9882353 ]],

        [[0.77254903, 0.8235294 , 0.9372549 ],
         [0.7647059 , 0.8156863 , 0.93333334],
         [0.7647059 , 0.8235294 , 0.9411765 ],
         ...,
         [0.90588236, 0.9529412 , 0.98039216],
         [0.9137255 , 0.9529412 , 0.98039216],
         [0.94509804, 0.9607843 , 0.99215686]],

        [[0.87058824, 0.8862745 , 0.9647059 ],
         [0.8352941 , 0.8627451 , 0.9490196 ],
         [0.81960785, 0.85882354, 0.9529412 ],
         ...,
         [0.9529412 , 0.98039216, 0.9843137 ],
         [0.95686275, 0.9764706 , 0.9843137 ],
         [0.98039216, 0.98039216, 0.99215686]],

        ...,

        [[0.28235295, 0.28627452, 0.3882353 ],
         [0.2784314 , 0.2901961 , 0.4       ]